# 01 — Data Cleaning

Preparation and merging of TETE labelling data with demographic and financial data for French intermunicipal authorities (EPCI).

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

list(DATA_DIR.glob("*"))

[PosixPath('../data/raw/criteres_repartition_csv.xlsx'),
 PosixPath('../data/raw/Copie de collectivites_engagees_labellisation.xlsx')]

## 1. Importing TETE data

In [3]:
tete_path = DATA_DIR / "Copie de collectivites_engagees_labellisation.xlsx"

tete = pd.read_excel(tete_path)

tete.head()

,collectivite_id,nom,siren,departement_code,region_code,etoiles_cae,etoiles_eci
0,2886,Malaunay,217604024,76,28,5,0
1,1518,Orvault,214401143,44,52,5,0
2,1262,Échirolles,213801517,38,84,5,0
3,1864,Lorient,215601212,56,53,5,0
4,435,La Rochelle,211703004,17,75,5,0


## 2. Preparing EPCI data

In [4]:
criteres_path = DATA_DIR / "criteres_repartition_csv.xlsx"

In [5]:
epci_criteria = pd.read_excel(
    criteres_path,
    sheet_name="Critères EPCI",
    header=3
)

epci_criteria.shape

(1289, 78)

In [6]:
selected_columns = [
    "Numéro SIREN EPCI",
    "Libellé EPCI",
    "Département siège de l'EPCI",
    "Nature juridique",
    "Régime fiscal",
    "Population DGF de l'année N",
    "Population INSEE de l'année N",
    "Revenu des EPCI",
    "Potentiel fiscal",
    "Potentiel fiscal par habitant",
    "CIF"
]

epci_criteria[selected_columns].head(10)

,Numéro SIREN EPCI,Libellé EPCI,Département siège de l'EPCI,Nature juridique,Régime fiscal,Population DGF de l'année N,Population INSEE de l'année N,Revenu des EPCI,Potentiel fiscal,Potentiel fiscal par habitant,CIF
0,200029999,CC RIVES DE L'AIN - PAYS DE CERDON,01,CC,FPU,15806,15156,248215641,5493274,347.543591,0.423406
1,200040350,CC BUGEY SUD,01,CC,FPU,37268,34938,572052395,15986818,428.969035,0.384833
2,200042497,CC DOMBES SAONE VALLEE,01,CC,FPU,42382,41975,856554882,14075842,332.118399,0.376276
3,200042935,COMMUNAUTÉ D’AGGLOMÉRATION HAUT-BUGEY AGGLOMÉR...,01,CA,FPU,67291,65428,1018955200,46929028,697.404229,0.416589
4,200069193,CC DE LA DOMBES,01,CC,FPU,42010,41461,761854989,12653931,301.212354,0.347817
5,200070118,CC VAL DE SAÔNE CENTRE,01,CC,FPU,21848,21470,399737320,4903029,224.415461,0.370987
6,200070555,CC DE LA VEYLE,01,CC,FPU,23993,23753,400103796,7514842,313.20977,0.267645
7,200071371,COMMUNAUTÉ DE COMMUNES BRESSE ET SAÔNE,01,CC,FPU,26617,26294,439466681,8883661,333.758913,0.410364
8,200071751,CA DU BASSIN DE BOURG-EN-BRESSE,01,CA,FPU,141941,139518,2301203267,68856336,485.105332,0.432454
9,240100610,CC DE LA COTIERE A MONTLUEL,01,CC,FPU,26083,25906,453492214,13026910,499.440632,0.448023


In [7]:
epci = epci_criteria[selected_columns].copy()

epci = epci.rename(columns={
    "Numéro SIREN EPCI": "siren",
    "Libellé EPCI": "nom_epci",
    "Département siège de l'EPCI": "departement",
    "Nature juridique": "nature_juridique",
    "Régime fiscal": "regime_fiscal",
    "Population DGF de l'année N": "population_dgf",
    "Population INSEE de l'année N": "population_insee",
    "Revenu des EPCI": "revenu",
    "Potentiel fiscal": "potentiel_fiscal",
    "Potentiel fiscal par habitant": "potentiel_fiscal_hab",
    "CIF": "cif"
})

epci.head()

,siren,nom_epci,departement,nature_juridique,regime_fiscal,population_dgf,population_insee,revenu,potentiel_fiscal,potentiel_fiscal_hab,cif
0,200029999,CC RIVES DE L'AIN - PAYS DE CERDON,01,CC,FPU,15806,15156,248215641,5493274,347.543591,0.423406
1,200040350,CC BUGEY SUD,01,CC,FPU,37268,34938,572052395,15986818,428.969035,0.384833
2,200042497,CC DOMBES SAONE VALLEE,01,CC,FPU,42382,41975,856554882,14075842,332.118399,0.376276
3,200042935,COMMUNAUTÉ D’AGGLOMÉRATION HAUT-BUGEY AGGLOMÉR...,01,CA,FPU,67291,65428,1018955200,46929028,697.404229,0.416589
4,200069193,CC DE LA DOMBES,01,CC,FPU,42010,41461,761854989,12653931,301.212354,0.347817


In [8]:
numeric_variables = [
    "population_dgf",
    "population_insee",
    "revenu",
    "potentiel_fiscal",
    "potentiel_fiscal_hab",
    "cif"
]

for col in numeric_variables:
    epci[col] = pd.to_numeric(epci[col], errors="coerce")

epci[numeric_variables].isna().sum()

population_dgf          28
population_insee        28
revenu                  34
potentiel_fiscal        34
potentiel_fiscal_hab    34
cif                     34
dtype: int64

## 3. Checking identifiers and merging

In [9]:
print("TETE:", tete["siren"].dtype, tete["siren"].nunique())
print("EPCI:", epci["siren"].dtype, epci["siren"].nunique())

print("\nTETE duplicates:", tete["siren"].duplicated().sum())
print("EPCI duplicates:", epci["siren"].duplicated().sum())

TETE: int64 577
EPCI: int64 1289

TETE duplicates: 0
EPCI duplicates: 0


In [10]:
matched_sirens = tete["siren"].isin(epci["siren"])

print("TETE SIRENs matched with EPCI:", matched_sirens.sum())
print("TETE SIRENs not matched:", (~matched_sirens).sum())

TETE SIRENs matched with EPCI: 425
TETE SIRENs not matched: 152


Of the 577 entries in the TETE dataset, 425 can be matched with the EPCI financial dataset using their SIREN identifier.

The analysis therefore focuses on these 425 matched intermunicipal authorities, for which both TETE ratings and demographic and financial characteristics are available.

In [11]:
analyse = tete.merge(
    epci,
    on="siren",
    how="inner"
)

analyse.shape

(425, 17)

In [12]:
analyse.head()

,collectivite_id,nom,siren,departement_code,region_code,etoiles_cae,etoiles_eci,nom_epci,departement,nature_juridique,regime_fiscal,population_dgf,population_insee,revenu,potentiel_fiscal,potentiel_fiscal_hab,cif
0,4785,CU de Dunkerque,245900428,59,32,5,1,CU DE DUNKERQUE,59,CU,FPU,200935.0,195297.0,2.808942e+09,358346393.0,1783.394595,0.496956
1,5307,CU du Grand Poitiers,200069854,86,75,5,3,GRAND POITIERS COMMUNAUTE URBAINE,86,CU,FPU,204749.0,201413.0,3.113395e+09,90320097.0,441.125949,0.587967
2,4209,Brest Métropole,242900314,29,53,5,2,BREST METROPOLE,29,METROPOLE,FPU,221575.0,217444.0,3.464207e+09,114816700.0,518.184362,0.575951
3,4434,Grenoble-Alpes-Métropole,200040715,38,84,5,3,GRENOBLE ALPES METROPOLE,38,METROPOLE,FPU,462979.0,455436.0,7.928277e+09,319917433.0,690.997719,0.357764
4,4378,Rennes Métropole,243500139,35,53,5,4,RENNES MÉTROPOLE,35,METROPOLE,FPU,491543.0,483199.0,8.484612e+09,274730740.0,558.914968,0.469388


## 4. Final checks and export

In [13]:
analyse.isna().sum()

collectivite_id         0
nom                     0
siren                   0
departement_code        0
region_code             0
etoiles_cae             0
etoiles_eci             0
nom_epci                0
departement             0
nature_juridique        0
regime_fiscal           0
population_dgf          0
population_insee        0
revenu                  0
potentiel_fiscal        0
potentiel_fiscal_hab    0
cif                     0
dtype: int64

In [14]:
analyse["siren"] = analyse["siren"].astype(str)

analyse.to_csv(
    "../data/processed/tete_epci_clean.csv",
    index=False
)